In [1]:
import hashlib
import secrets
import math
import cupy as cp
import numpy as np
import time
import hmac

In [2]:
N = 32
W = 16

LOG_W = int(math.log2(W))

L1 = math.ceil((8 * N) / LOG_W)
L2 = math.floor(math.log(L1 * (W - 1), W)) + 1
L = L1 + L2

print(f"L1 = {L1}")
print(f"L2 = {L2}")
print(f"L  = {L}")

L1 = 64
L2 = 3
L  = 67


In [3]:
def sha256(data):
    return hashlib.sha256(data).digest()


def prf(seed, index):
    return hmac.new(seed, index.to_bytes(4, "big"), hashlib.sha256).digest()

def chain_wots_plus(x, start, steps, pub_seed, chain_idx):
    value = x
    for s in range(start, start + steps):
        addr  = chain_idx * W + s
        mask  = prf(pub_seed, addr)
        xored = bytes(a ^ b for a, b in zip(value, mask))
        value = sha256(xored)
    return value

In [4]:
def to_base_w(data, w, out_len):
    value = int.from_bytes(data, "big")
    digits = []
    for _ in range(out_len):
        digits.append(value % w)
        value //= w
    digits.reverse()
    return digits

In [5]:
def compute_checksum(msg_digits):
    checksum = sum((W - 1 - d) for d in msg_digits)
    digits = []
    for _ in range(L2):
        digits.append(checksum % W)
        checksum //= W
    digits.reverse()
    return digits

In [6]:
def message_to_coefficients(message):

    digest = sha256(message)

    msg_digits = to_base_w(
        digest,
        W,
        L1
    )

    checksum_digits = compute_checksum(
        msg_digits
    )

    return msg_digits + checksum_digits

In [7]:
def keygen():
    pub_seed = secrets.token_bytes(N)
    sk = [secrets.token_bytes(N) for _ in range(L)]
    pk = [chain_wots_plus(sk[i], 0, W-1, pub_seed, i) for i in range(L)]
    return pub_seed, sk, pk

In [8]:
def sign(message, sk, pub_seed):
    a = message_to_coefficients(message)
    return [chain_wots_plus(sk[i], 0, a[i], pub_seed, i) for i in range(L)]

In [9]:
def verify(message, signature, pk, pub_seed):
    a = message_to_coefficients(message)
    for i in range(L):
        candidate = chain_wots_plus(signature[i], a[i], W-1-a[i], pub_seed, i)
        if candidate != pk[i]:
            return False
    return True

In [10]:
message = b"Hello WOTS+"
pub_seed, sk, pk = keygen()
sig = sign(message, sk, pub_seed)
print("Valid msg  :", verify(message, sig, pk, pub_seed))   # True
print("Wrong msg  :", verify(b"Hello WOTS", sig, pk, pub_seed))  # False

Valid msg  : True
Wrong msg  : False


In [11]:
bad_message = b"Hello WOTS"

print(
    verify(
        bad_message,
        sig,
        pk,
        pub_seed
    )
)

False


In [12]:
NTEST = 100

start = time.perf_counter()

for _ in range(NTEST):
    keygen()

end = time.perf_counter()

print(
    "Average KeyGen:",
    (end-start)/NTEST*1000,
    "ms"
)

Average KeyGen: 27.34193194999989 ms


In [13]:
pub_seed, sk, pk = keygen()

message = b"benchmark"

start = time.perf_counter()

for _ in range(NTEST):
    sign(message, sk, pub_seed)

end = time.perf_counter()

print(
    "Average Sign:",
    (end-start)/NTEST*1000,
    "ms"
)

Average Sign: 7.695769219999988 ms


In [14]:
sig = sign(message, sk, pub_seed)

start = time.perf_counter()

for _ in range(NTEST):
    verify(message, sig, pk, pub_seed)

end = time.perf_counter()

print(
    "Average Verify:",
    (end-start)/NTEST*1000,
    "ms"
)

Average Verify: 7.770705549999946 ms


In [15]:
signature_size = L * N
public_key_size = L * N
private_key_size = L * N

print("Private key:", private_key_size, "bytes")
print("Public key:", public_key_size, "bytes")
print("Signature:", signature_size, "bytes")

Private key: 2144 bytes
Public key: 2144 bytes
Signature: 2144 bytes


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# GPU IMPLEMENTATION

In [17]:
props = cp.cuda.runtime.getDeviceProperties(0)

try:
    print(props["name"].decode())
except:
    print(props["name"])

Tesla T4


In [18]:
gpu_hash_kernel = cp.RawKernel(r'''
extern "C" __global__
void gpu_hash_chain(unsigned int* data, int rounds)
{
    int idx = blockDim.x * blockIdx.x + threadIdx.x;

    unsigned int v = data[idx];

    for(int i = 0; i < rounds; i++)
    {
        v ^= (v << 13);
        v ^= (v >> 17);
        v ^= (v << 5);
    }

    data[idx] = v;
}
''', 'gpu_hash_chain')

In [19]:
num_signatures = 10000

num_chains = num_signatures * L

gpu_data = cp.random.randint(
    0,
    2**32,
    size=num_chains,
    dtype=cp.uint32
)

threads = 256
blocks = (num_chains + threads - 1) // threads

cp.cuda.Stream.null.synchronize()

start = time.perf_counter()

gpu_hash_kernel(
    (blocks,),
    (threads,),
    (gpu_data, np.int32(15))

)

cp.cuda.Stream.null.synchronize()

elapsed = time.perf_counter() - start

print(f"Chains: {num_chains:,}")
print(f"GPU Time: {elapsed:.6f} sec")
print(f"Throughput: {num_chains/elapsed:,.0f} chains/sec")

Chains: 670,000
GPU Time: 0.042847 sec
Throughput: 15,637,098 chains/sec


In [20]:
import cupy as cp
import numpy as np

GPU_KERNEL_SRC = r"""
typedef unsigned char      uint8_t;
typedef unsigned int       uint32_t;
typedef unsigned long long uint64_t;

#define ROTRIGHT(a,b) (((a) >> (b)) | ((a) << (32-(b))))
#define CH(x,y,z)  (((x) & (y)) ^ (~(x) & (z)))
#define MAJ(x,y,z) (((x) & (y)) ^ ((x) & (z)) ^ ((y) & (z)))
#define EP0(x) (ROTRIGHT(x,2)  ^ ROTRIGHT(x,13) ^ ROTRIGHT(x,22))
#define EP1(x) (ROTRIGHT(x,6)  ^ ROTRIGHT(x,11) ^ ROTRIGHT(x,25))
#define SIG0(x)(ROTRIGHT(x,7)  ^ ROTRIGHT(x,18) ^ ((x) >> 3))
#define SIG1(x)(ROTRIGHT(x,17) ^ ROTRIGHT(x,19) ^ ((x) >> 10))

__constant__ uint32_t K256[64] = {
    0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,
    0x923f82a4,0xab1c5ed5,0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,
    0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf174,0xe49b69c1,0xefbe4786,
    0x0fc19dc6,0x240ca1cc,0x2de92c6f,0x4a7484aa,0x5cb0a9dc,0x76f988da,
    0x983e5152,0xa831c66d,0xb00327c8,0xbf597fc7,0xc6e00bf3,0xd5a79147,
    0x06ca6351,0x14292967,0x27b70a85,0x2e1b2138,0x4d2c6dfc,0x53380d13,
    0x650a7354,0x766a0abb,0x81c2c92e,0x92722c85,0xa2bfe8a1,0xa81a664b,
    0xc24b8b70,0xc76c51a3,0xd192e819,0xd6990624,0xf40e3585,0x106aa070,
    0x19a4c116,0x1e376c08,0x2748774c,0x34b0bcb5,0x391c0cb3,0x4ed8aa4a,
    0x5b9cca4f,0x682e6ff3,0x748f82ee,0x78a5636f,0x84c87814,0x8cc70208,
    0x90befffa,0xa4506ceb,0xbef9a3f7,0xc67178f2
};

__device__ void sha256_32(const uint8_t* in, uint8_t* out) {
    uint32_t m[64];
    for (int i = 0; i < 8; i++)
        m[i] = ((uint32_t)in[i*4]   << 24) | ((uint32_t)in[i*4+1] << 16)
             | ((uint32_t)in[i*4+2] <<  8) |  (uint32_t)in[i*4+3];
    m[8] = 0x80000000u;
    for (int i = 9; i < 15; i++) m[i] = 0;
    m[15] = 256u;
    for (int i = 16; i < 64; i++)
        m[i] = SIG1(m[i-2]) + m[i-7] + SIG0(m[i-15]) + m[i-16];

    uint32_t a=0x6a09e667u, b=0xbb67ae85u, c=0x3c6ef372u, d=0xa54ff53au;
    uint32_t e=0x510e527fu, f=0x9b05688cu, g=0x1f83d9abu, h=0x5be0cd19u;
    for (int i = 0; i < 64; i++) {
        uint32_t t1 = h + EP1(e) + CH(e,f,g) + K256[i] + m[i];
        uint32_t t2 = EP0(a) + MAJ(a,b,c);
        h=g; g=f; f=e; e=d+t1;
        d=c; c=b; b=a; a=t1+t2;
    }
    uint32_t H[8] = {
        a+0x6a09e667u, b+0xbb67ae85u, c+0x3c6ef372u, d+0xa54ff53au,
        e+0x510e527fu, f+0x9b05688cu, g+0x1f83d9abu, h+0x5be0cd19u
    };
    for (int i = 0; i < 8; i++) {
        out[i*4]   = (H[i] >> 24) & 0xff;
        out[i*4+1] = (H[i] >> 16) & 0xff;
        out[i*4+2] = (H[i] >>  8) & 0xff;
        out[i*4+3] =  H[i]        & 0xff;
    }
}

__device__ void prf_mask(uint64_t seed_word, int addr, uint8_t* mask) {
    uint64_t state = seed_word ^ ((uint64_t)addr * 0x9e3779b97f4a7c15ULL);
    for (int b = 0; b < 32; b += 8) {
        state ^= state >> 12;
        state ^= state << 25;
        state ^= state >> 27;
        uint64_t v = state * 0x2545F4914F6CDD1DULL;
        for (int j = 0; j < 8 && b+j < 32; j++)
            mask[b+j] = (uint8_t)(v >> (j * 8));
    }
}

extern "C" __global__
void wots_plus_batch_chain(
    const uint8_t* __restrict__ inputs,
    uint8_t*       __restrict__ outputs,
    const int*     __restrict__ steps,
    const int*     __restrict__ chain_offsets,
    unsigned long long          seed_word,
    int                         num_chains
) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    if (idx >= num_chains) return;

    uint8_t buf[32], tmp[32], mask[32];
    for (int b = 0; b < 32; b++)
        buf[b] = inputs[idx * 32 + b];

    int start = chain_offsets[idx];
    int end   = start + steps[idx];

    for (int s = start; s < end; s++) {
        int addr = idx * W + s;
        prf_mask(seed_word, addr, mask);
        for (int b = 0; b < 32; b++) tmp[b] = buf[b] ^ mask[b];
        sha256_32(tmp, buf);
    }

    for (int b = 0; b < 32; b++)
        outputs[idx * 32 + b] = buf[b];
}
"""

GPU_KERNEL_SRC = GPU_KERNEL_SRC.replace(
    'typedef unsigned char      uint8_t;',
    f'typedef unsigned char      uint8_t;\n#define W {W}'
)

THREADS = 256

props = cp.cuda.runtime.getDeviceProperties(0)
try:    print("GPU:", props["name"].decode())
except: print("GPU:", props["name"])

kernel = cp.RawKernel(GPU_KERNEL_SRC, 'wots_plus_batch_chain',
                      options=('--std=c++14',))
print("Kernel compiled OK")

GPU: Tesla T4
Kernel compiled OK


In [21]:
def _seed_to_uint64(pub_seed):
    return int.from_bytes(pub_seed[:8], "big")

def _gpu_run_chains(inputs_np, steps_np, offsets_np, pub_seed):
    num_chains = inputs_np.shape[0]
    d_in      = cp.asarray(inputs_np.flatten())
    d_out     = cp.zeros(num_chains * 32, dtype=cp.uint8)
    d_steps   = cp.asarray(steps_np,  dtype=cp.int32)
    d_offsets = cp.asarray(offsets_np, dtype=cp.int32)
    seed_word = np.uint64(_seed_to_uint64(pub_seed))
    blocks    = (num_chains + THREADS - 1) // THREADS

    kernel(
        (blocks,), (THREADS,),
        (d_in, d_out, d_steps, d_offsets, seed_word, np.int32(num_chains))
    )
    cp.cuda.Stream.null.synchronize()
    return cp.asnumpy(d_out).reshape(num_chains, 32)


def gpu_keygen_batch(n=1000):
    pub_seed   = secrets.token_bytes(N)
    num_chains = n * L
    sk_np      = np.frombuffer(
                     secrets.token_bytes(num_chains * 32), dtype=np.uint8
                 ).reshape(num_chains, 32).copy()

    steps   = np.full(num_chains, W - 1, dtype=np.int32)
    offsets = np.zeros(num_chains,        dtype=np.int32)
    pk_np   = _gpu_run_chains(sk_np, steps, offsets, pub_seed)

    return pub_seed, sk_np, pk_np


def gpu_sign_batch(messages, sk_np, pub_seed):
    n          = len(messages)
    num_chains = n * L
    steps      = np.zeros(num_chains, dtype=np.int32)
    offsets    = np.zeros(num_chains, dtype=np.int32)

    for ki, msg in enumerate(messages):
        coeffs = message_to_coefficients(msg)
        for ci in range(L):
            steps  [ki * L + ci] = coeffs[ci]
            offsets[ki * L + ci] = 0

    return _gpu_run_chains(sk_np, steps, offsets, pub_seed)


def gpu_verify_batch(messages, sig_np, pk_np, pub_seed):
    n          = len(messages)
    num_chains = n * L
    steps      = np.zeros(num_chains, dtype=np.int32)
    offsets    = np.zeros(num_chains, dtype=np.int32)

    for ki, msg in enumerate(messages):
        coeffs = message_to_coefficients(msg)
        for ci in range(L):
            steps  [ki * L + ci] = W - 1 - coeffs[ci]
            offsets[ki * L + ci] = coeffs[ci]

    candidates = _gpu_run_chains(sig_np, steps, offsets, pub_seed)

    results = []
    for ki in range(n):
        match = np.array_equal(
            candidates[ki*L : (ki+1)*L],
            pk_np     [ki*L : (ki+1)*L]
        )
        results.append(bool(match))
    return results

In [22]:
N_BATCH = 1000
NTEST   = 50

t0 = time.perf_counter()
for _ in range(NTEST): keygen()
cpu_keygen_ms = (time.perf_counter() - t0) / NTEST * 1000

pub_seed_cpu, sk_cpu, pk_cpu = keygen()
msg_bench = b"benchmark"

t0 = time.perf_counter()
for _ in range(NTEST): sign(msg_bench, sk_cpu, pub_seed_cpu)
cpu_sign_ms = (time.perf_counter() - t0) / NTEST * 1000

sig_cpu = sign(msg_bench, sk_cpu, pub_seed_cpu)
t0 = time.perf_counter()
for _ in range(NTEST): verify(msg_bench, sig_cpu, pk_cpu, pub_seed_cpu)
cpu_verify_ms = (time.perf_counter() - t0) / NTEST * 1000

gpu_keygen_batch(10)
cp.cuda.Stream.null.synchronize()

t0 = time.perf_counter()
pub_seed_g, sk_g, pk_g = gpu_keygen_batch(N_BATCH)
gpu_keygen_ms = (time.perf_counter() - t0) * 1000 / N_BATCH

msgs = [b"benchmark"] * N_BATCH
t0 = time.perf_counter()
sigs_g = gpu_sign_batch(msgs, sk_g, pub_seed_g)
gpu_sign_ms = (time.perf_counter() - t0) * 1000 / N_BATCH

t0 = time.perf_counter()
results = gpu_verify_batch(msgs, sigs_g, pk_g, pub_seed_g)
gpu_verify_ms = (time.perf_counter() - t0) * 1000 / N_BATCH

print(f"All {N_BATCH} GPU verifications passed: {all(results)}")


print()
print(f"{'Operation':<10}  {'CPU (ms)':>10}  {'GPU (ms)':>10}  {'Speedup':>9}")
print(f"{'-'*10}  {'-'*10}  {'-'*10}  {'-'*9}")
for name, cpu_t, gpu_t in [
    ("KeyGen",  cpu_keygen_ms,  gpu_keygen_ms),
    ("Sign",    cpu_sign_ms,    gpu_sign_ms),
    ("Verify",  cpu_verify_ms,  gpu_verify_ms),
]:
    print(f"{name:<10}  {cpu_t:>10.3f}  {gpu_t:>10.3f}  {cpu_t/gpu_t:>8.1f}x")

print(f"\nGPU batch size : {N_BATCH} keypairs per launch")
print(f"Total chains   : {N_BATCH * L:,}  ({W-1} hashes each)")

print()
print(f"{'Scheme':<14}  {'SK (B)':>8}  {'PK (B)':>8}  {'Sig (B)':>8}")
print(f"{'-'*14}  {'-'*8}  {'-'*8}  {'-'*8}")
print(f"{'W-OTS+':<14}  {L*N:>8}  {L*N:>8}  {L*N:>8}")
print(f"{'ECDSA P-256':<14}  {32:>8}  {64:>8}  {64:>8}")
print(f"\nOverhead vs ECDSA:  SK x{(L*N)//32}  PK x{(L*N)//64}  Sig x{(L*N)//64}")
print(f"\nSHA-256 calls per op (W-OTS+, L={L}, W={W}):")
print(f"  KeyGen : {L*(W-1)}  ({L} chains x {W-1} hashes)")
print(f"  Sign   : ~{L*((W-1)//2)}")
print(f"  Verify : ~{L*((W-1)//2)}")

All 1000 GPU verifications passed: True

Operation     CPU (ms)    GPU (ms)    Speedup
----------  ----------  ----------  ---------
KeyGen           7.282       0.014     528.9x
Sign             3.045       0.038      79.2x
Verify           3.900       0.060      64.9x

GPU batch size : 1000 keypairs per launch
Total chains   : 67,000  (15 hashes each)

Scheme            SK (B)    PK (B)   Sig (B)
--------------  --------  --------  --------
W-OTS+              2144      2144      2144
ECDSA P-256           32        64        64

Overhead vs ECDSA:  SK x67  PK x33  Sig x33

SHA-256 calls per op (W-OTS+, L=67, W=16):
  KeyGen : 1005  (67 chains x 15 hashes)
  Sign   : ~469
  Verify : ~469
